In [4]:
!pip install -q pdfplumber PyPDF2 sentence-transformers faiss-cpu transformers accelerate einops


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 97.6 MB/s eta 0:00:00


**Contextual retrieval + generation (RAG)**

Features:
- Loads PDFs from a folder and extracts text (uses pdfplumber or PyPDF2 fallback).
- Chunks text and computes embeddings with sentence-transformers (all-MiniLM-L6-v2).
- Stores vectors in FAISS (in-memory, free) for retrieval.
- Uses a small, free LLM (google/flan-t5-base) from Hugging Face Transformers for generation.
  This model runs on CPU reasonably in Colab; for better quality you can switch to `flan-t5-large` if you have a GPU.
- No cloud paid APIs required. Works with only pip installs and Hugging Face model downloads.

Usage (Colab):
1. Upload your PDFs into a folder (e.g., /content/docs) or mount Google Drive and point `PDF_FOLDER` to it.
2. Run the cells sequentially. After index creation you can call `ask(query)` interactively.

Limitations & notes:
- flan-t5-base is a small model — answers may be shorter and less detailed than large LLMs.
- Quality depends on chunking and retrieval; tune chunk_size and overlap.
- For more accurate embeddings you can try larger sentence-transformers but they will be slower.

In [5]:
# ---------------------------
# Install dependencies (run this cell in Colab)
# ---------------------------
# !pip install -q pdfplumber PyPDF2 sentence-transformers faiss-cpu transformers accelerate einops

# ---------------------------
# Imports
# ---------------------------
import os
import glob
import math
from typing import List, Tuple

try:
    import pdfplumber
except Exception:
    pdfplumber = None

from sentence_transformers import SentenceTransformer
import numpy as np

In [7]:
# FAISS (CPU)
import faiss

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

In [8]:
# ---------------------------
# Configuration
# ---------------------------
PDF_FOLDER = "/content/docs"  # change to your folder containing PDFs
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # lightweight, free
GENERATION_MODEL_NAME = "google/flan-t5-base"  # small, free LLM suitable for CPU

In [9]:
# Chunking params
CHUNK_SIZE = 500  # characters per chunk (tune) - Reduced from 800
CHUNK_OVERLAP = 100  # overlap between chunks - Reduced from 200
TOP_K = 5  # number of chunks to retrieve

In [10]:
# FAISS index dim will be determined after loading embeddings
INDEX = None
DOC_CHUNKS: List[str] = []  # parallel list to embeddings

In [11]:
# ---------------------------
# Utilities: PDF text extraction
# ---------------------------

def extract_text_from_pdf(path: str) -> str:
    """Try pdfplumber, fall back to PyPDF2 text extraction."""
    text_parts = []
    if pdfplumber is not None:
        try:
            with pdfplumber.open(path) as pdf:
                for page in pdf.pages:
                    page_text = page.extract_text() or ""
                    text_parts.append(page_text)
            return "\n".join(text_parts)
        except Exception as e:
            print(f"pdfplumber failed for {path}: {e}")

    # fallback to PyPDF2
    try:
        import PyPDF2
        with open(path, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            for p in reader.pages:
                page_text = p.extract_text() or ""
                text_parts.append(page_text)
        return "\n".join(text_parts)
    except Exception as e:
        print(f"PyPDF2 fallback failed for {path}: {e}")
        return ""


def load_pdfs_from_folder(folder: str) -> Tuple[List[str], List[str]]:
    """Loads PDFs, returns list of filenames and corresponding text."""
    pdf_paths = sorted(glob.glob(os.path.join(folder, "**", "*.pdf"), recursive=True))
    texts = []
    for p in pdf_paths:
        print("Reading:", p)
        t = extract_text_from_pdf(p)
        texts.append(t)
    return pdf_paths, texts

In [12]:
# ---------------------------
# Text chunking
# ---------------------------

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> List[str]:
    """Chunk by characters (keeps sentences intact reasonably by breaking on spaces)."""
    text = text.replace('\r', ' ')
    if not text:
        return []
    chunks = []
    start = 0
    length = len(text)
    while start < length:
        end = start + chunk_size
        if end >= length:
            chunks.append(text[start:].strip())
            break
        # try to avoid cutting mid-word: backtrack to last space within a small window
        window = text[start:end]
        last_space = window.rfind(' ')
        if last_space == -1 or last_space < int(chunk_size * 0.6):
            last_space = end
        chunk = text[start:start + last_space].strip()
        chunks.append(chunk)
        start = start + last_space - overlap
        if start < 0:
            start = 0
    return [c for c in chunks if len(c) > 30]

In [13]:
# ---------------------------
# Build index
# ---------------------------

def build_index_from_folder(folder: str):
    global INDEX, DOC_CHUNKS

    # load texts
    paths, texts = load_pdfs_from_folder(folder)
    print(f"Loaded {len(paths)} PDFs")

    # chunk and keep metadata
    chunks = []
    chunk_sources = []  # source filename for each chunk
    for path, text in zip(paths, texts):
        file_chunks = chunk_text(text)
        chunks.extend(file_chunks)
        chunk_sources.extend([os.path.basename(path)] * len(file_chunks))
    print(f"Created {len(chunks)} chunks")

    # compute embeddings
    embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)
    embeddings = embedder.encode(chunks, show_progress_bar=True, convert_to_numpy=True)

    # create FAISS index
    dim = embeddings.shape[1]
    INDEX = faiss.IndexFlatIP(dim)  # cosine via inner-product with normalized vectors

    # normalize embeddings for cosine similarity
    faiss.normalize_L2(embeddings)
    INDEX.add(embeddings)

    DOC_CHUNKS = chunks
    # store metadata as parallel arrays (could be enhanced to store titles/filenames)
    print("Index built. Total vectors:", INDEX.ntotal)

In [14]:
# ---------------------------
# Retrieval
# ---------------------------

def retrieve(query: str, top_k: int = TOP_K) -> List[Tuple[int, float]]:
    """Return list of (chunk_index, score)"""
    embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)
    q_emb = embedder.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_emb)
    distances, indices = INDEX.search(q_emb, top_k)
    # distances are inner products; higher is better
    results = []
    for idx, score in zip(indices[0], distances[0]):
        if idx == -1:
            continue
        results.append((int(idx), float(score)))
    return results

In [15]:
# ---------------------------
# Generation (RAG) using Flan-T5
# ---------------------------

print("Loading generation model (this may take a while on first run)...")
# Use device_map='auto' if GPU present; on CPU it will still work but slower
try:
    tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL_NAME)
    gen_model = AutoModelForSeq2SeqLM.from_pretrained(GENERATION_MODEL_NAME)
    generator = pipeline("text2text-generation", model=gen_model, tokenizer=tokenizer, device_map="auto")
except Exception:
    # fallback to CPU pipeline
    tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL_NAME)
    gen_model = AutoModelForSeq2SeqLM.from_pretrained(GENERATION_MODEL_NAME, device_map={"": "cpu"})
    generator = pipeline("text2text-generation", model=gen_model, tokenizer=tokenizer)


SYSTEM_PROMPT = (
    "You are a helpful assistant. Use the provided context (delimited by <CONTEXT>...</CONTEXT>) to answer the question. "
    "If the answer is not contained in the context, say you don't know and avoid inventing facts. Be concise but complete."
)


def generate_answer(question: str, retrieved_chunks: List[Tuple[int, float]]) -> str:
    """Create a prompt with top retrieved chunks and generate an answer."""
    if not retrieved_chunks:
        return "I couldn't find any relevant context to answer that question."

    # assemble context
    assembled = []
    for idx, score in retrieved_chunks:
        assembled.append(DOC_CHUNKS[idx])
    context = "\n\n---\n\n".join(assembled)

    prompt = (
        f"{SYSTEM_PROMPT}\n\n<CONTEXT>\n{context}\n</CONTEXT>\n\nQuestion: {question}\nAnswer:")

    # send to generator — control max_length as needed
    out = generator(prompt, max_length=512, do_sample=False)
    answer = out[0]['generated_text']
    # sometimes flan's output may include the question again; try to clean by removing prompt prefix
    if prompt in answer:
        answer = answer.split(prompt)[-1].strip()
    return answer


Loading generation model (this may take a while on first run)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Device set to use cuda:0


In [16]:
# ---------------------------
# Convenience: build index and ask
# ---------------------------

def build_and_save_index(folder: str = PDF_FOLDER):
    build_index_from_folder(folder)
    print("Index ready. You can now call ask(query)")


def ask(question: str, top_k: int = TOP_K) -> str:
    if INDEX is None:
        return "Index is not built yet. Run build_and_save_index(folder) first."
    retrieved = retrieve(question, top_k=top_k)
    print("Retrieved top chunks (idx, score):", retrieved)
    answer = generate_answer(question, retrieved)
    return answer

In [17]:
# ---------------------------
# Example usage (for Colab cell):
# 1) Put PDFs into /content/docs or set PDF_FOLDER
# 2) Run: build_and_save_index(PDF_FOLDER)
# 3) Then: print(ask("What is the main contribution of the paper?"))
# ---------------------------

if __name__ == "__main__":
    # quick self-test: if /content/docs doesn't exist, download a sample PDF to test
    if not os.path.exists(PDF_FOLDER) or len(glob.glob(os.path.join(PDF_FOLDER, "*.pdf"))) == 0:
        print("No PDFs found in", PDF_FOLDER)
        os.makedirs(PDF_FOLDER, exist_ok=True)
        sample_pdf = os.path.join(PDF_FOLDER, "attention_is_all_you_need.pdf")
        try:
            import urllib.request
            url = "https://arxiv.org/pdf/1706.03762.pdf"
            print("Downloading sample PDF from", url)
            urllib.request.urlretrieve(url, sample_pdf)
            print("Downloaded sample to", sample_pdf)
        except Exception as e:
            print("Could not download sample PDF:", e)

    print("Building index from folder:", PDF_FOLDER)
    build_and_save_index(PDF_FOLDER)

    # sample question
    q = "What is the main idea of the document?"
    print("Question:", q)
    print("Answer:", ask(q))

Building index from folder: /content/docs
Reading: /content/docs/ACK645989310240723.pdf
Reading: /content/docs/Sanjeev_Ranjan_Resume.pdf
Loaded 2 PDFs
Created 24 chunks


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Index built. Total vectors: 24
Index ready. You can now call ask(query)
Question: What is the main idea of the document?


Token indices sequence length is longer than the specified maximum sequence length for this model (729 > 512). Running this sequence through the model will result in indexing errors


Retrieved top chunks (idx, score): [(23, 0.21876247227191925), (22, 0.18983455002307892), (19, 0.18273143470287323), (20, 0.1470913589000702), (6, 0.1423618048429489)]


Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: Artificial Intelligence & Data Science Projects TextProcessing&ComputerVision: Led project for scalable self-supervised pipelines for unstructured text & image data using clustering + supervised models; delivered 92% accuracy with NLP workflows (Word2Vec, PCA, Logistics Regression) and computer vision pipelines (convolutions, K-Means, t-SNE).


In [ ]:
print("Enter your questions about the documents. Type 'quit' to exit.")
while True:
    user_input = input("You: ")
    if user_input.lower() == 'quit':
        break
    answer = ask(user_input)
    print("Bot:", answer)

print("Chat session ended.")